In [ ]:
import os, sys
from ast import literal_eval
import pickle
from pathlib import Path
import glob

import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from tqdm import tqdm
import yaml

from torch.nn.functional import one_hot
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import torchvision
from torchvision.ops import StochasticDepth
from typing import List
import PIL
from PIL import Image
from glob import glob
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import cv2

c:\Users\xuant\anaconda3\envs\goAI\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from torchvision import transforms

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize(320),
        transforms.RandomCrop(300),       
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        transforms.ColorJitter(
            brightness=0.1,
            contrast=0.1,
            saturation=0.05,
            hue=0.01
        ),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),

    'val': transforms.Compose([
        transforms.Resize(320),
        transforms.CenterCrop(300),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
}


In [4]:
trainset = datasets.ImageFolder(r'C:\WoodAI\data\data_split\train',transform=data_transforms['train'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=12,
                                          shuffle=True, num_workers=4, drop_last=False)
#testset = dataset.CUB(root='./CUB_200_2011', is_train=False, data_len=None)
testset = datasets.ImageFolder(r'C:\WoodAI\data\data_split\val',transform=data_transforms['val'])
testloader = torch.utils.data.DataLoader(testset, batch_size=6,
                                         shuffle=False, num_workers=4, drop_last=False)

In [5]:
dataset_sizes = {"train": len(trainset) ,
                "val": len(testset),
               }
data_loaders = {"train": trainloader,
                "val" : testloader                    
               }

In [6]:
dataset_sizes

{'train': 7480, 'val': 886}

In [7]:
trainset.classes

['Bach dan (Lang Son)',
 'Bach dan (Thai Nguyen)',
 'Gao',
 'Gioi',
 'Keo lai',
 'Keo tai tuong',
 'Lat hoa',
 'Lim xanh',
 'Mo (Thai Nguyen)',
 'Mo Tuyen Quang',
 'Que',
 'Tech',
 'Thong',
 'Xoan ta (Dai Tu)',
 'Xoan ta (Dinh Hoa)',
 'other']

In [8]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
from torchvision import models
model = models.efficientnet_v2_s(pretrained=True)

for params in model.parameters():
    params.requires_grad = True

# Change the final classification head.

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3), 
    nn.Linear(in_features=1280, out_features=len(trainset.classes))
)
model = model.to(device, memory_format=torch.channels_last)
print("Model loaded and moved to device:", next(model.parameters()).device)
dummy_input = torch.randn(1, 3, 300, 300).to(device, memory_format=torch.channels_last)
model.eval()
with torch.no_grad():
    _ = model(dummy_input)

print("📦 After dummy input:")
print("🔍 CUDA memory allocated:", round(torch.cuda.memory_allocated(0)/1024**2, 2), "MB")
print("🔍 CUDA memory reserved:", round(torch.cuda.memory_reserved(0)/1024**2, 2), "MB")


c:\Users\xuant\anaconda3\envs\goAI\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\xuant\anaconda3\envs\goAI\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_V2_S_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model loaded and moved to device: cuda:0
📦 After dummy input:
🔍 CUDA memory allocated: 87.91 MB
🔍 CUDA memory reserved: 98.0 MB


In [9]:
# import time
# import copy
# def train_model(model, criterion, optimizer, scheduler, data_loaders, dataset_sizes, device, num_epochs):
#     since = time.time()

#     best_model_wts = copy.deepcopy(model.state_dict())
#     best_acc = 0.0
#     print("Model device:", next(model.parameters()).device)

#     for epoch in range(num_epochs):
#         print(f'Epoch {epoch}/{num_epochs - 1}')
#         print('-' * 10)

#         for phase in ['train', 'val']:
#             if phase == 'train':
#                 model.train()
#             else:
#                 model.eval()

#             running_loss = 0.0
#             running_corrects = 0
#             loop = tqdm(data_loaders[phase], desc=f"{phase} [{epoch+1}/{num_epochs}]")
#             for inputs, labels in loop:
#                 inputs = inputs.to(device)
#                 labels = labels.to(device)

#                 optimizer.zero_grad()

#                 with torch.set_grad_enabled(phase == 'train'):
#                     outputs = model(inputs)
#                     _, preds = torch.max(outputs, 1)
#                     loss = criterion(outputs, labels)

#                     if phase == 'train':
#                         loss.backward()
#                         optimizer.step()

#                 running_loss += loss.item() * inputs.size(0)
#                 running_corrects += torch.sum(preds == labels.data)

#             if phase == 'train' and scheduler is not None:
#                 scheduler.step()

#             epoch_loss = running_loss / dataset_sizes[phase]
#             epoch_acc = running_corrects.double() / dataset_sizes[phase]

#             print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

#             # Lưu mô hình tốt nhất theo validation accuracy
#             if phase == 'val' and epoch_acc > best_acc:
#                 best_acc = epoch_acc
#                 best_model_wts = copy.deepcopy(model.state_dict())

#                 # Tạo thư mục lưu nếu chưa có
#                 save_dir = "save_model"
#                 os.makedirs(save_dir, exist_ok=True)
#                 filename_model =f"Go_v2_epoch_{epoch}.pth"

#                 # Lưu state_dict thay vì model toàn bộ để tránh lỗi
#                 torch.save(best_model_wts, filename_model)
#                 print(f'✅ Model saved at epoch {epoch} to {filename_model}')

#         print()

#     time_elapsed = time.time() - since
#     print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
#     print(f'Best val Acc: {best_acc:.4f}')

#     # Load lại trọng số mô hình tốt nhất
#     model.load_state_dict(best_model_wts)
#     return model
import time
import copy
import os
from tqdm import tqdm
import torch.nn.functional as F
import torch

def mixup_data(x, y, alpha=0.4):
    '''MixUp'''
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    '''CutMix'''
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size, _, H, W = x.size()
    index = torch.randperm(batch_size).to(x.device)
    y_a, y_b = y, y[index]

    rx = torch.randint(W, (1,)).item()
    ry = torch.randint(H, (1,)).item()
    rw = int(W * torch.sqrt(1 - lam).item())
    rh = int(H * torch.sqrt(1 - lam).item())
    x1 = max(rx - rw // 2, 0)
    y1 = max(ry - rh // 2, 0)
    x2 = min(rx + rw // 2, W)
    y2 = min(ry + rh // 2, H)

    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    return x, y_a, y_b, lam

def train_model(model, criterion, optimizer, scheduler, data_loaders, dataset_sizes, device,
                num_epochs, patience=5, use_mixup=True, use_cutmix=True,
                mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.5):
    
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    best_acc = 0.0
    early_stop_counter = 0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0
            loop = tqdm(data_loaders[phase], desc=f"{phase} [{epoch+1}/{num_epochs}]")

            for inputs, labels in loop:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    # Augmentation trong TRAIN
                    if phase == 'train':
                        do_mix = torch.rand(1).item() < mix_prob
                        if do_mix and use_mixup:
                            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
                        elif do_mix and use_cutmix:
                            inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha=cutmix_alpha)
                        else:
                            targets_a, targets_b, lam = labels, labels, 1.0

                        outputs = model(inputs)
                        loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                        _, preds = torch.max(outputs, 1)
                        acc_targets = targets_a
                    else:
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                        _, preds = torch.max(outputs, 1)
                        acc_targets = labels

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == acc_targets.data)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # === EARLY STOPPING ===
            if phase == 'val':
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    early_stop_counter = 0

                    os.makedirs("save_model", exist_ok=True)
                    filename_model = f"save_model/Go_v4_epoch_{epoch}.pth"
                    torch.save(best_model_wts, filename_model)
                    print(f'✅ Model saved: {filename_model}')
                else:
                    early_stop_counter += 1
                    print(f'⏳ No improvement. Early stop counter: {early_stop_counter}/{patience}')

        if early_stop_counter >= patience:
            print(f'⛔ Early stopping at epoch {epoch}')
            break

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model



In [ ]:
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

model_ft = train_model(model, criterion, optimizer_ft, exp_lr_scheduler,
                       data_loaders, dataset_sizes, device, num_epochs=40, patience=7, use_mixup=True,
                       mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.5)